In [23]:
import json
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from collections import deque

In [24]:
random.seed(42)

In [25]:
def parse_conversations(json_file_path):
    with open(json_file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    parsed_data = []

    for sample in data:
        conversation_list = []

        for message in sample["conversation"]:
            conversation_list.append(
                {
                    "u": message["user"],
                    "a": message["assistant"],
                    "label": message["label"],
                }
            )

        parsed_data.append(conversation_list)

    return parsed_data

In [26]:
def parse_attack_dataset(json_file_path):
    with open(json_file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    parsed_data = []

    for sample in data:
        conversation = []

        for turn in sample["turns"]:
            conversation.append(
                {
                    "u": turn["attack_message"],
                    "a": turn["target_response"],
                    "label": turn["judge_result"],
                }
            )

        parsed_data.append(conversation)

    return parsed_data

In [27]:
def parse_benign(json_file_path):
    parsed_data = []

    with open(json_file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            sample = json.loads(line)

            conversation = []
            convs = sample["conversations"]

            i = 0
            while i < len(convs) - 1:
                if convs[i]["from"] == "human" and convs[i + 1]["from"] == "gpt":
                    conversation.append({"u": convs[i]["value"],
                            "a": convs[i + 1]["value"],
                            "label": 0, })
                    i += 2
                else:
                    i += 1

            parsed_data.append(conversation)

    return parsed_data

In [28]:
src1=parse_conversations("./../data/multi_turn_conversations_from_opposite_day_08.json")
src2=parse_conversations("./../data/multi_turn_conversations_from_crescendomation_08.json")
src3=parse_attack_dataset("./../data/conversation_multiTurn_scaleAI.json")


In [29]:
src4=parse_benign("./../data/benign_data.jsonl")

In [30]:
src=[]
src.extend(src1)
src.extend(src2)
src.extend(src3)
src.extend(src4)

with open("./../data/combined_conversations.json", "w", encoding="utf-8") as f:
    json.dump(src, f, ensure_ascii=False, indent=4)



In [31]:
with open("./../data/combined_conversations.json", "r", encoding="utf-8") as f:
    combined_conversations = json.load(f)

In [32]:
from sentence_transformers import SentenceTransformer
emb_model = SentenceTransformer('all-mpnet-base-v2') 

In [ ]:
src_embedded = []

for conv_id, conversation in enumerate(combined_conversations):
    convo = []    
    for turn_id, turn in enumerate(conversation):
        turn_embedded = {} 
        u_emb = emb_model.encode(turn["u"])
        a_emb = emb_model.encode(turn["a"])
        turn_embedded["u"] = u_emb
        turn_embedded["a"] = a_emb
        turn_embedded["label"] = turn["label"]
        turn_embedded["conv_id"] = conv_id
        turn_embedded["turn_id"] = turn_id
        convo.append(turn_embedded)
        
    src_embedded.append(convo)

# Model

In [ ]:
class stateSpaceModel(nn.Module):
    def __init__(self,state_dim, input_dim, hidden_dim1,hidden_dim2, output_dim):
        super(stateSpaceModel, self).__init__()
        # F(x_{t-1},u_t)=>x_t (Transition model)
        self.Fxu = nn.Sequential(
            nn.Linear(state_dim + input_dim, hidden_dim1),
            nn.ReLU(),
            nn.Linear(hidden_dim1, hidden_dim2),
            nn.ReLU(),
            nn.Linear(hidden_dim2, state_dim)
        )

        # G(x_t,u_t)=>z_t (Observation model)
        self.Gxu = nn.Sequential(
            nn.Linear(state_dim + input_dim, hidden_dim1),
            nn.ReLU(),
            nn.Linear(hidden_dim1, hidden_dim2),
            nn.ReLU(),
            nn.Linear(hidden_dim2, output_dim)
        )
        
    def forward(self, x_prev, u):
        # gets xt
        ux_prev = torch.cat([u,x_prev], dim=-1)
        x_curr = self.Fxu(ux_prev)

        # gets zt
        ux_curr = torch.cat([u,x_curr], dim=-1)
        zt = self.Gxu(ux_curr)
        return x_curr, zt

In [ ]:
state_dim = 768    # x_t
input_dim = 768    # u_t
output_dim = 768   # z_t
hidden_dim_ssm1 = 1200  
hidden_dim_ssm2 = 900  

model=stateSpaceModel(state_dim=state_dim, input_dim=input_dim, hidden_dim1=hidden_dim_ssm1, \
                      hidden_dim2=hidden_dim_ssm2, output_dim=output_dim)

In [ ]:
checkpoint = torch.load("./../..//NBF/ssm_models/ssm_model_exp5/models_best_ssm.pth", map_location=torch.device('cpu'))
model.load_state_dict(checkpoint["ssm"])
model.eval()

In [ ]:
src_convos=[]
with torch.no_grad(): 
    for convo in src_embedded:
        single_conversation=[]
        x_prev = torch.zeros(state_dim)
        for turn in convo:
            u = torch.tensor(turn['u'], dtype=torch.float32)
            x_curr, zt = model(x_prev, u)
            single_conversation.append({"x_t":x_curr,
                                        "ut":u,
                                        "zt_approx":zt,
                                        "y":turn['label'],
                                        "conv_id":turn["conv_id"],
                                        "turn_id":turn["turn_id"],
                                        })
            x_prev = x_curr
        src_convos.append(single_conversation)

In [ ]:
torch.save(src_convos, "./all_conversations.pt")

In [ ]:
def feature_engineering(x_t,u_t,x_prev,x_prev_4step_back=None):
    features = []

    # drift features
    state_drift = torch.norm(x_t -x_prev).item()
    features.append(state_drift)

    state_input_distance = torch.norm(x_t -u_t).item()
    features.append(state_input_distance)
        
    long_term_state_drift = torch.norm(
        x_t - x_prev_4step_back
    ).item()
    features.append(long_term_state_drift)

    # similarity features
    state_similarity = F.cosine_similarity(
        x_t.unsqueeze(0),
        x_prev.unsqueeze(0)
    ).item()
    features.append(state_similarity)

    state_input_similarity = F.cosine_similarity(
        x_t.unsqueeze(0),
        u_t.unsqueeze(0)
    ).item()
    features.append(state_input_similarity)


    long_term_state_similarity = F.cosine_similarity(
        x_t.unsqueeze(0),
        x_prev_4step_back.unsqueeze(0)
    ).item()
    features.append(long_term_state_similarity)


    return torch.tensor(features, dtype=torch.float32)

In [ ]:
import numpy as np 

In [ ]:
X_vecs = []
X_features = []
y = []

for convo in src_convos:
    first_x = convo[0]["x_t"]
    first_u = convo[0]["ut"]

    x_prev = torch.zeros_like(convo[0]["x_t"])
    x_history = deque(maxlen=4)

    for turn in convo:
        X_vecs.append(torch.concat([turn["x_t"], turn["ut"]], dim=0))

        if len(x_history) == 4:
            x_prev_4step_back = x_history[0]
        else:
            x_prev_4step_back = first_x

        handcrafted = feature_engineering(turn["x_t"],turn["ut"],x_prev,x_prev_4step_back).numpy()
        ids = np.array([turn["conv_id"], turn["turn_id"]],dtype=np.int32)
        handcrafted = np.concatenate([handcrafted, ids])

        X_features.append(handcrafted)
        y.append(turn["y"])

        x_history.append(turn["x_t"])
        x_prev = turn["x_t"]

In [ ]:
X_vecs_np = torch.stack(X_vecs).cpu().numpy()

feature_columns = [
    "state_drift", "state_input_distance", "long_term_state_drift",
    "state_similarity", "state_input_similarity",
    "long_term_state_similarity", "conv_id", "turn_id"
]

X_features_df = pd.DataFrame(X_features, columns=feature_columns)

df = pd.concat([
    pd.DataFrame(X_vecs_np),
    X_features_df
], axis=1)

df["y"] = y

In [ ]:
df2 = pd.read_csv("./features_before_selection.csv")

In [ ]:
joined_df= pd.merge(df, df2, on=["conv_id", "turn_id"], how="inner")

In [ ]:
assert len(joined_df) == len(df) == len(df2)
assert (joined_df["y"]==joined_df["label"]).all()

In [ ]:
df.to_csv("./ssm_all_features.csv", index=False)